In [13]:
import warnings
warnings.filterwarnings("ignore")

import optuna
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

In [14]:
MASTER = "/kaggle/input/notebooks/shri7ul/03-feature-engineering-ipynb/master_feature_engineered.parquet"

master = pd.read_parquet(MASTER)

print(master.shape)

(35072, 122)


In [15]:
def objective(trial):

    params = {

        "objective": "binary:logistic",
        "eval_metric": "logloss",

        "tree_method": "hist",

        "random_state": 42,

        "n_estimators": 5000,

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.05,
            log=True,
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            6,
            10,
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10,
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.7,
            1.0,
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            0.8,
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5,
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5,
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.5,
            10,
        ),

        "early_stopping_rounds":300,
    }

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(**params)

        model.fit(

            X_train,
            y_train,

            eval_set=[(X_valid, y_valid)],

            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:,1]

        oof[valid_idx] = pred

    score = log_loss(y, oof)

    return score

In [16]:
TARGET = "is_correct"

DROP_COLUMNS = [

    "response_id",
    "session_id",

    "learning_objective",
    "learning_objective_id",

    "transcript",
    "student_text",
    "tutor_text",
    "background_text",

    TARGET,
]

X = master.drop(columns=DROP_COLUMNS)

y = master[TARGET].astype(int)

print(X.shape)
print(y.shape)

(35072, 113)
(35072,)


In [17]:
cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    X[col] = X[col].astype("category").cat.codes

print(cat_cols.tolist())

['objective_family']


In [18]:
print(type(X))
print(type(y))

print(X.shape)
print(y.shape)

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
(35072, 113)
(35072,)


In [19]:
study = optuna.create_study(
    direction="minimize",
    study_name="semantic_xgb",
)

study.optimize(

    objective,

    n_trials=30,

    show_progress_bar=True,
)

[I 2026-08-04 02:59:24,206] A new study created in memory with name: semantic_xgb


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-08-04 03:02:13,610] Trial 0 finished with value: 0.5438812458377402 and parameters: {'learning_rate': 0.019272211097824295, 'max_depth': 10, 'min_child_weight': 1, 'subsample': 0.939766964534546, 'colsample_bytree': 0.5524133051847101, 'gamma': 0.034146127491127, 'reg_alpha': 0.43063018485313076, 'reg_lambda': 5.9706103957483405}. Best is trial 0 with value: 0.5438812458377402.
[I 2026-08-04 03:03:16,872] Trial 1 finished with value: 0.5458571171761242 and parameters: {'learning_rate': 0.03689101818073302, 'max_depth': 10, 'min_child_weight': 5, 'subsample': 0.8753752680364386, 'colsample_bytree': 0.7868959921825884, 'gamma': 4.35243682536435, 'reg_alpha': 3.509080543602387, 'reg_lambda': 3.041893768335137}. Best is trial 0 with value: 0.5438812458377402.
[I 2026-08-04 03:04:58,613] Trial 2 finished with value: 0.5451447548432945 and parameters: {'learning_rate': 0.01183077485364891, 'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.9849178179254221, 'colsample_bytree': 0.

In [20]:
print("="*60)

print("Best LogLoss")

print(study.best_value)

print()

print("Best Params")

for k,v in study.best_params.items():

    print(f"{k:20s} : {v}")

print("="*60)

Best LogLoss
0.5428791502080297

Best Params
learning_rate        : 0.022620902773803953
max_depth            : 8
min_child_weight     : 9
subsample            : 0.9349106863970573
colsample_bytree     : 0.5867674888298766
gamma                : 0.00326704915093734
reg_alpha            : 2.2229715588419947
reg_lambda           : 6.40159602102026


In [1]:
optuna.visualization.plot_optimization_history(study)

NameError: name 'optuna' is not defined

In [22]:
optuna.visualization.plot_param_importances(study)